# IRED Latent Diffusion — Full Training Notebook

End-to-end pipeline for the **gensis.md §8** experiment:

1. **Milestone 1** — train the AttentionPool on top of a frozen BART so that `Decoder(Pool(Encoder(A))) ≈ A`.
2. **Milestone 2** — freeze the AE, train the IRED energy network on `(z_q, z_a)` pairs.
3. **Milestone 3** — sweep `inner_steps` and plot accuracy vs. test-time compute.

Defaults match the *Practical defaults that survived first-run debugging* section of gensis.md (full answer mode, `K=32`, cosine β schedule, `weight_decay=0.01` on the EBM, SDPA math kernel).

Runs end-to-end on a single GPU. On CPU, lower `STEPS_AE` / `STEPS_EBM` for a smoke test only — real convergence needs a GPU.

## Colab / fresh-environment setup

No-op when run locally with the repo already installed. On Colab (or any clean env where the `ired` package isn't importable), this cell:

1. Detects whether the `ired` package is available.
2. If not, gets the repo into the current working directory. Three options — uncomment the one you want:
    - **clone** from a remote (fastest if the code is on GitHub),
    - **mount Google Drive** and `cd` into a copy you've already pushed there, or
    - manually **upload** the `ired/` directory + `pyproject.toml` via Colab's file panel.
3. `pip`-installs the runtime deps (Colab doesn't ship `uv`).
4. Adds the repo root to `sys.path` so `from ired.* import ...` resolves.

Recommended GPU: Colab Free T4 fits `bart-base` at `batch_size=16, max_a_length=256`. Sessions disconnect after ~90 min idle — checkpoints land in `checkpoints/ae/` and `checkpoints/ebm/`; you can mount Drive and point the checkpoint dirs there to survive disconnects.

In [1]:
import sys, os, subprocess, importlib.util, shutil

IN_COLAB = "google.colab" in sys.modules
HAS_IRED = importlib.util.find_spec("ired") is not None

if IN_COLAB and not HAS_IRED:
    # ---- 1) GET THE CODE INTO CWD --------------------------------------
    # Pick ONE of the three options below.

    # Option A: clone from GitHub. Replace the URL with your fork.
    # Always pulls the freshest commit: if the repo dir exists, fetch + hard-reset
    # to origin/HEAD; otherwise clone fresh.
    REPO_URL = "https://github.com/DoubleAmp3rsand/ired-reasoning"
    REPO_DIR = "ired-reasoning"
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--prune"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/HEAD"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "clean", "-fd"], check=True)
    else:
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    head_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
    print(f"checked out {REPO_DIR} @ {head_sha}")


    # Option C: upload `ired/` + `pyproject.toml` via the Files sidebar,
    # then leave cwd as /content (the default).

    if not os.path.isdir("ired"):
        raise RuntimeError(
            "Colab detected but no 'ired/' package found in cwd. "
            "Uncomment one of the options above (clone / Drive / upload) and re-run."
        )

    # ---- 2) INSTALL DEPS (Colab usually has torch + transformers already) ----
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "torch>=2.5.0", "transformers>=4.45.0", "datasets>=2.20.0",
         "tqdm", "sentencepiece", "protobuf>=4.25.0", "accelerate>=0.34.0"],
        check=True,
    )

    # ---- 3) MAKE `ired` IMPORTABLE -----------------------------------------
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())
    HAS_IRED = importlib.util.find_spec("ired") is not None

if HAS_IRED:
    print(f"ired package importable from: {os.getcwd()}")
else:
    print("ired package NOT importable — fix the setup options above before continuing.")

checked out ired-reasoning @ 44f3c09
ired package importable from: /content/ired-reasoning


## Configuration

Edit this cell to override hyperparameters. Everything downstream reads from `CONFIG`.

In [2]:
import os, time, math, torch

# ─── Ablation switch ──────────────────────────────────────────────────────────
# Toggle this between "decoder" and "resampler" and re-run from m1-build to
# train each variant. Checkpoints auto-route to separate dirs based on this
# so the two runs don't overwrite each other.
POOL_TYPE = "decoder"   # or "resampler"
# ──────────────────────────────────────────────────────────────────────────────

CONFIG = {
    # model — must match the HF AE checkpoint's training config so
    # `ae.load_ae(ckpt['ae'])` doesn't error on shape mismatch.
    "model_name":    "facebook/bart-base",
    "k":             256,
    "pool_layers":   2,
    "pool_heads":    8,
    "pool_type":     POOL_TYPE,
    "ebm_layers":    6,
    "ebm_heads":     8,
    "ebm_ff_mult":   4,

    # LD4LG compression/reconstruction split (see gensis §7.3)
    "d_ae":          None,      # None = d_model. Matches checkpoint (saved as -1).
    "recon_layers":  4,
    "recon_heads":   8,

    # AE weights — milestone 2 downloads these from Hugging Face instead of
    # depending on a local milestone-1 run.
    "ae_hf_repo":    "roy-W/ired-ae",
    "ae_hf_file":    "ae_step28000.pt",
    "ae_hf_revision": "main",

    # data — AE on OpenWebText (broad natural language) + a Python slice from
    # CodeSearchNet (2019 scrape, predates MBPP/HumanEval so no eval-set leakage).
    # EBM on MBPP (Python code); held-out eval on HumanEval. Both correctness
    # signals come from executing decoded code in a subprocess.
    "owt_samples":      50000,
    "owt_min_chars":    200,
    "owt_max_chars":    4000,
    "code_mix_ratio":   0.3,    # 0 = pure OWT (legacy); 0.3 = 70% OWT + 30% code
    "code_samples":     50000,
    "code_min_chars":   100,
    "code_max_chars":   4000,
    "mixed_length":     200000, # virtual epoch length for the OWT+code mix
    "mbpp_config":      "full", # 'full' = 974 examples; 'sanitized' = 427 cleaned
    "max_q_length":     512,    # HumanEval prompts (signature + docstring) can be long
    "max_a_length":     512,    # covers MBPP solutions and HumanEval canonical bodies
    "eval_n_examples":  80,     # per-corpus, per eval cycle

    # diffusion
    "timesteps":     10,
    "inner_steps":   5,
    "beta_schedule": "cosine",
    "opt_step_size": 1.0,
    "x_start_clamp": 5.0,
    "envelope_sf":   None,
    "nce_scale":     1.0,
    "supervise_nce": True,

    # decoder-CE auxiliary (see gensis §7.2)
    "decoder_aux_weight": 1.0,
    "decoder_aux_t_max":  2,

    # random-negative anchor: forces E(real) + margin < E(N(0,I)) so the EBM
    # cannot leave random latents at the same energy as gold. Gated to low t
    # (rand_neg_t_max) because at high t, q(z_t|z_a) and N(0,I) overlap —
    # forcing separation there crowds out gradient signal for the eps
    # prediction and stalls denoiser learning.
    "rand_neg_weight": 1.0,
    "rand_neg_margin": 1.0,
    "rand_neg_t_max":  3,       # with timesteps=10, anchor only fires for t<3

    # training
    "batch_size":    4,
    "grad_accum_steps": 8,      # effective batch = batch_size * grad_accum_steps
    "steps_ae":     15000,
    "steps_ebm":     4000,
    "lr_ae":         1e-4,
    "wd_ae":         0.01,
    "warmup_ae":     1000,
    "lr_ebm":        3e-4,
    "weight_decay":  0.01,
    "log_every":     50,
    "eval_every":    500,

    # IO — ae_dir tagged by pool_type so the two ablation runs don't collide.
    "ae_dir":        f"checkpoints/ae_{POOL_TYPE}",
    "ebm_dir":       f"checkpoints/ebm_{POOL_TYPE}",
    "device":        "cuda" if torch.cuda.is_available() else "cpu",
    "num_workers":   2,
    "seed":          0,
}

torch.manual_seed(CONFIG["seed"])
os.makedirs(CONFIG["ae_dir"],  exist_ok=True)
os.makedirs(CONFIG["ebm_dir"], exist_ok=True)
print(f"device: {CONFIG['device']}")
print(f"pool_type: {CONFIG['pool_type']}   ae_dir: {CONFIG['ae_dir']}   ebm_dir: {CONFIG['ebm_dir']}")


device: cuda
pool_type: decoder   ae_dir: checkpoints/ae_decoder   ebm_dir: checkpoints/ebm_decoder


In [3]:
from torch.optim import AdamW
from torch.utils.data import DataLoader

from ired.autoencoder import FrozenBartAutoencoder
from ired.data import (
    CodeSearchNetDataset,
    HumanEvalDataset,
    MBPPDataset,
    MixedDataset,
    OpenWebTextDataset,
    collate,
    verify_code,
)
from ired.diffusion import GaussianLatentDiffusion
from ired.energy_net import DiffusionWrapper, EnergyTransformer
from ired.train_autoencoder import eval_reconstruction
from ired.train_diffusion import eval_corpus


## Data

Builds three data sources: OpenWebText train loader (AE pretraining), MBPP train loader (EBM training), and MBPP/HumanEval test sets (used for both AE round-trip and EBM end-to-end eval; correctness goes through `verify_code`, which executes decoded Python against each example's test fixture in a subprocess).


In [4]:
# AE training corpus: OpenWebText + optional CodeSearchNet Python slice.
# CodeSearchNet is a 2019 GitHub scrape that predates MBPP (2021) and
# HumanEval (2021) — contamination of the EBM eval sets is structurally ruled
# out by the temporal cutoff.
print("loading OpenWebText train slice (streaming)...")
owt_train_ds = OpenWebTextDataset(
    max_samples=CONFIG["owt_samples"],
    min_chars=CONFIG["owt_min_chars"],
    max_chars=CONFIG["owt_max_chars"],
    seed=CONFIG["seed"],
)
print(f"  materialized {len(owt_train_ds)} OWT docs")

if CONFIG["code_mix_ratio"] > 0.0:
    print("loading CodeSearchNet Python slice (streaming)...")
    code_train_ds = CodeSearchNetDataset(
        max_samples=CONFIG["code_samples"],
        min_chars=CONFIG["code_min_chars"],
        max_chars=CONFIG["code_max_chars"],
        seed=CONFIG["seed"],
    )
    print(f"  materialized {len(code_train_ds)} code fns")
    ae_train_ds = MixedDataset(
        datasets=[owt_train_ds, code_train_ds],
        weights=[1.0 - CONFIG["code_mix_ratio"], CONFIG["code_mix_ratio"]],
        length=CONFIG["mixed_length"],
    )
    print(
        f"  mixed train dataset: virtual length {len(ae_train_ds)} "
        f"({(1.0 - CONFIG['code_mix_ratio']):.0%} OWT + {CONFIG['code_mix_ratio']:.0%} code)"
    )
else:
    ae_train_ds = owt_train_ds
    print("  (code_mix_ratio=0; training on pure OpenWebText)")

# Held-out OWT slice for AE byte-recon sanity (different seed → effectively disjoint).
owt_eval_ds = OpenWebTextDataset(
    max_samples=CONFIG["eval_n_examples"] * 4,
    min_chars=CONFIG["owt_min_chars"],
    max_chars=CONFIG["owt_max_chars"],
    seed=CONFIG["seed"] + 1,
)
print(f"  materialized {len(owt_eval_ds)} OWT eval docs")

# EBM training corpus: MBPP.
print("loading MBPP train split...")
mbpp_train_ds = MBPPDataset(
    split="train", config=CONFIG["mbpp_config"], seed=CONFIG["seed"],
)

# Test splits for AE and EBM evaluation (MBPP test + HumanEval held-out).
mbpp_test_ds = MBPPDataset(
    split="test", config=CONFIG["mbpp_config"],
    max_samples=CONFIG["eval_n_examples"], seed=CONFIG["seed"],
)
he_test_ds = HumanEvalDataset(max_samples=CONFIG["eval_n_examples"], seed=CONFIG["seed"])

ae_train_loader = DataLoader(
    ae_train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
    collate_fn=collate, num_workers=CONFIG["num_workers"], drop_last=True,
)
ebm_train_loader = DataLoader(
    mbpp_train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
    collate_fn=collate, num_workers=CONFIG["num_workers"], drop_last=True,
)
print(f"AE  train: {len(ae_train_ds)} examples  |  AE  eval slice: {len(owt_eval_ds)}")
print(f"EBM train: MBPP ({len(mbpp_train_ds)} examples, config={CONFIG['mbpp_config']})")
print(f"Eval: MBPP test={len(mbpp_test_ds)}  HumanEval={len(he_test_ds)}")


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Skylion007/openwebtext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Skylion007/openwebtext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


loading OpenWebText train slice (streaming)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

...

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Skylion007/openwebtext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Skylion007/openwebtext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  materialized 50000 code fns
  mixed train dataset: virtual length 200000 (70% OWT + 30% code)


Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

  materialized 320 OWT eval docs
loading MBPP train split...
AE  train: 200000 examples  |  AE  eval slice: 320
EBM train: MBPP (374 examples, config=full)
Eval: MBPP test=80  HumanEval=80


## Milestone 1 — Train the AttentionPool + Reconstructor on OpenWebText

Frozen BART + a learned `K` attention pool + reconstruction net. Trains by reconstruction CE: `Decoder(Recon(Pool(Encoder(text)))) → text` on OpenWebText.

**Decision rule (gensis §9):** MBPP pass-rate ≥ 80% AND HumanEval pass-rate ≥ 80% on their test splits, even though the AE only ever trained on OpenWebText. Pass-rate (not byte-exact match) is the right bar — Python tokens are higher entropy than English prose, and a single mis-rendered operator can flake a whole test. If both code bars fail by a wide margin, the OWT-trained latent space doesn't transfer to Python; recovery is to add a code slice to AE pretraining or swap BART for CodeT5.

### Ablation: `pool_type` ∈ {`"decoder"`, `"resampler"`}

The pool has two implementations; A/B test which trains better on this task. Protocol:

1. Set `POOL_TYPE = "decoder"` in the config cell. Run from `m1-build` through `m1-train`. Checkpoints land in `checkpoints/ae_decoder/`.
2. Restart kernel (or re-run from `m1-build`). Set `POOL_TYPE = "resampler"`. Run again. Checkpoints land in `checkpoints/ae_resampler/`.
3. Compare the `[eval]` lines — `mbpp_pass`, `humaneval_pass`, `loss`, and trajectory shape.

**What we expect:**
- `decoder`: nn.TransformerDecoderLayer pool. Empirically reliable baseline.
- `resampler`: Flamingo-faithful Perceiver Resampler with shared LayerNorm on the KV path and gated residuals (α=0 init). If LD4LG's design transfers, this should match or beat `decoder`.


In [ ]:
ae = FrozenBartAutoencoder(
    model_name=CONFIG["model_name"],
    k=CONFIG["k"],
    pool_layers=CONFIG["pool_layers"],
    pool_heads=CONFIG["pool_heads"],
    pool_type=CONFIG["pool_type"],
    d_ae=CONFIG["d_ae"],
    recon_layers=CONFIG["recon_layers"],
    recon_heads=CONFIG["recon_heads"],
).to(CONFIG["device"])
ae.train()

ae_params = ae.trainable_parameters()
print(f"trainable AE params: {sum(p.numel() for p in ae_params):,}  (pool + recon)")
print(f"pool_type={ae.pool_type}   d_model={ae.d_model}   d_ae={ae.d_ae}")

In [ ]:
from torch.optim.lr_scheduler import LambdaLR

opt = AdamW(ae_params, lr=CONFIG["lr_ae"], weight_decay=CONFIG["wd_ae"])

# Linear warmup over warmup_ae steps, then constant at lr_ae.
def _lr_lambda(step: int) -> float:
    w = CONFIG["warmup_ae"]
    if w <= 0:
        return 1.0
    return min(1.0, float(step + 1) / float(w))
sched = LambdaLR(opt, _lr_lambda)
print(f"optimizer: AdamW(lr={CONFIG['lr_ae']}, wd={CONFIG['wd_ae']})  warmup={CONFIG['warmup_ae']}")

step = 0
train_iter = iter(ae_train_loader)
t0 = time.time()

def _snip(s, n=180):
    s = s.replace("\n", " ⏎ ")
    return s if len(s) <= n else s[:n] + "…"

while step < CONFIG["steps_ae"]:
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(ae_train_loader)
        batch = next(train_iter)

    z = ae.encode_to_latents(batch["answer"], CONFIG["device"], max_length=CONFIG["max_a_length"])
    loss = ae.decode_loss(z, batch["answer"], CONFIG["device"], max_length=CONFIG["max_a_length"])

    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(ae_params, 1.0)
    opt.step()
    sched.step()

    step += 1
    if step % CONFIG["log_every"] == 0:
        cur_lr = opt.param_groups[0]["lr"]
        print(f"[ae] step {step:6d} | loss {loss.item():.4f} | lr {cur_lr:.2e} | {time.time()-t0:.1f}s")

    if step % CONFIG["eval_every"] == 0 or step == CONFIG["steps_ae"]:
        ev_owt = eval_reconstruction(
            ae, owt_eval_ds, "byte", CONFIG["device"], CONFIG["max_a_length"],
            n_examples=CONFIG["eval_n_examples"], batch_size=CONFIG["batch_size"],
        )
        ev_mbpp = eval_reconstruction(
            ae, mbpp_test_ds, "code", CONFIG["device"], CONFIG["max_a_length"],
            n_examples=CONFIG["eval_n_examples"], batch_size=CONFIG["batch_size"],
        )
        ev_he = eval_reconstruction(
            ae, he_test_ds, "code", CONFIG["device"], CONFIG["max_a_length"],
            n_examples=CONFIG["eval_n_examples"], batch_size=CONFIG["batch_size"],
        )
        print(
            f"  [eval] owt={ev_owt['acc']:.3f} (n={ev_owt['n']})  "
            f"mbpp_pass={ev_mbpp['acc']:.3f} (n={ev_mbpp['n']})  "
            f"humaneval_pass={ev_he['acc']:.3f} (n={ev_he['n']})"
        )
        print(
            f"    losses: owt={ev_owt['loss']:.3f}  mbpp={ev_mbpp['loss']:.3f}  humaneval={ev_he['loss']:.3f}"
        )
        for name, ev in [("mbpp", ev_mbpp), ("humaneval", ev_he)]:
            if ev["samples"]:
                g, p = ev["samples"][0]
                print(f"    [{name}] gold: {_snip(g)}")
                print(f"    [{name}] pred: {_snip(p)}")
        ckpt = {
            "ae": ae.state_dict_ae(),
            "config": CONFIG,
            "step": step,
            "owt_recon_acc": ev_owt["acc"],
            "mbpp_pass": ev_mbpp["acc"],
            "humaneval_pass": ev_he["acc"],
        }
        torch.save(ckpt, os.path.join(CONFIG["ae_dir"], f"ae_step{step}.pt"))
        torch.save(ckpt, os.path.join(CONFIG["ae_dir"], "ae_latest.pt"))

print(f"\nMilestone 1 done. Final mbpp_pass={ev_mbpp['acc']:.3f} humaneval_pass={ev_he['acc']:.3f}")


## Milestone 2 — Train the IRED energy network

Freeze everything in the AE (BART + pool). Train only the `EnergyTransformer`.

**Per-step monitoring:**
- `mse, nce` — denoising MSE + NCE energy contrast loss.
- `e_real, e_fake` — absolute energies of clean vs. mined-fake samples. Watch the **ratio**, not the absolutes.
- `eps_scale = ||ε̂|| / ||noise||` — should stay near 1.0. Drift signals head-magnitude inflation that breaks DDPM regardless of MSE.

**Per-eval monitoring:**
- `ae_recon_acc` — should stay at Milestone 1's level; if it drops, the load path broke.
- `ebm_acc(inner=N)` vs `ebm_acc(inner=0)` — isolates whether `opt_step` is helping.
- `mse_z, std_zs, corr_z` — direction (`corr_z`) vs. magnitude (`std_zs`) of where `z_sampled` lands relative to `z_a`.

In [5]:
# Load pretrained AE weights from Hugging Face (skips milestone-1 dependency).
# Reads the checkpoint's saved CONFIG to rebuild the AE with matching
# architecture — this avoids shape-mismatch errors when notebook CONFIG drifts
# from the checkpoint and removes any dependence on stale kernel state.
from huggingface_hub import hf_hub_download

print(f"downloading {CONFIG['ae_hf_repo']}/{CONFIG['ae_hf_file']} from HF...")
ae_ckpt_path = hf_hub_download(
    repo_id=CONFIG["ae_hf_repo"],
    filename=CONFIG["ae_hf_file"],
    revision=CONFIG["ae_hf_revision"],
    repo_type="model",
)
print(f"  cached at: {ae_ckpt_path}")

ae_ckpt = torch.load(ae_ckpt_path, map_location=CONFIG["device"], weights_only=False)
ck_cfg = ae_ckpt.get("config", {})

# Pull AE architecture from the checkpoint; fall back to notebook CONFIG when absent.
# d_ae stored as -1 in the training script means "use d_model" (i.e. None to the API).
ck_d_ae = ck_cfg.get("d_ae")
if ck_d_ae is not None and ck_d_ae <= 0:
    ck_d_ae = None

ae_arch = {
    "model_name":   ck_cfg.get("model_name",   CONFIG["model_name"]),
    "k":            ck_cfg.get("k",            CONFIG["k"]),
    "pool_layers":  ck_cfg.get("pool_layers",  CONFIG["pool_layers"]),
    "pool_heads":   ck_cfg.get("pool_heads",   CONFIG["pool_heads"]),
    "pool_type":    ck_cfg.get("pool_type",    CONFIG["pool_type"]),
    "d_ae":         ck_d_ae if "d_ae" in ck_cfg else CONFIG["d_ae"],
    "recon_layers": ck_cfg.get("recon_layers", CONFIG["recon_layers"]),
    "recon_heads":  ck_cfg.get("recon_heads",  CONFIG["recon_heads"]),
}
print("AE architecture from checkpoint:")
for k, v in ae_arch.items():
    nb_v = CONFIG.get(k) if k != "d_ae" else CONFIG.get("d_ae")
    flag = "" if v == nb_v else f"  (CONFIG had {nb_v!r})"
    print(f"  {k:14s} = {v!r}{flag}")

# Sync the notebook CONFIG so downstream cells (m2-build, m2-train, eval) see
# the same K / d_ae the checkpoint was trained with.
for k, v in ae_arch.items():
    if k == "model_name":
        continue
    CONFIG[k] = v

ae = FrozenBartAutoencoder(**ae_arch).to(CONFIG["device"])
ae.load_ae(ae_ckpt["ae"])
print(
    f"loaded AE checkpoint  step={ae_ckpt.get('step')}  "
    f"mbpp_pass={ae_ckpt.get('mbpp_pass')}  humaneval_pass={ae_ckpt.get('humaneval_pass')}"
)

ae.eval()
for p in ae.parameters():
    p.requires_grad_(False)
print("AE frozen.")

downloading roy-W/ired-ae/ae_step28000.pt from HF...
  cached at: /root/.cache/huggingface/hub/models--roy-W--ired-ae/snapshots/1a55568e51608d5e1564285e28daec11deef2f9f/ae_step28000.pt
AE architecture from checkpoint:
  model_name     = 'facebook/bart-base'
  k              = 256
  pool_layers    = 2
  pool_heads     = 8
  pool_type      = 'decoder'
  d_ae           = None
  recon_layers   = 4
  recon_heads    = 8


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

/content/ired-reasoning/ired/autoencoder.py:243: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.layers = nn.TransformerEncoder(layer, num_layers=n_layers)


loaded AE checkpoint  step=28000  mbpp_pass=0.09375  humaneval_pass=0.0
AE frozen.


In [6]:
d_diff = ae.d_ae  # EBM diffuses in the diffusion-space dimension (post-pool, pre-recon)
ebm = EnergyTransformer(
    d_model=d_diff,
    k=CONFIG["k"],
    n_layers=CONFIG["ebm_layers"],
    n_heads=CONFIG["ebm_heads"],
    dim_ff_mult=CONFIG["ebm_ff_mult"],
).to(CONFIG["device"])
wrapper = DiffusionWrapper(ebm).to(CONFIG["device"])

diffusion = GaussianLatentDiffusion(
    model=wrapper,
    latent_shape=(CONFIG["k"], d_diff),
    timesteps=CONFIG["timesteps"],
    beta_schedule=CONFIG["beta_schedule"],
    opt_step_size=CONFIG["opt_step_size"],
    loss_scale=CONFIG["nce_scale"],
    supervise_energy_landscape=CONFIG["supervise_nce"],
    x_start_clamp=CONFIG["x_start_clamp"],
    envelope_sf=CONFIG["envelope_sf"],
    decoder_aux_weight=CONFIG["decoder_aux_weight"],
    decoder_aux_t_max=CONFIG["decoder_aux_t_max"],
    rand_neg_weight=CONFIG["rand_neg_weight"],
    rand_neg_margin=CONFIG["rand_neg_margin"],
    rand_neg_t_max=CONFIG["rand_neg_t_max"],
).to(CONFIG["device"])
diffusion.train()

if CONFIG["decoder_aux_weight"] > 0:
    diffusion.set_decoder_loss_fn(
        lambda x, txt: ae.decode_loss(x, txt, CONFIG["device"], max_length=CONFIG["max_a_length"])
    )

print(f"ebm params: {sum(p.numel() for p in ebm.parameters()):,}  (diffuses in d_ae={d_diff}, layers={CONFIG['ebm_layers']})")
if CONFIG["rand_neg_weight"] > 0:
    print(
        f"rand-neg anchor: weight={CONFIG['rand_neg_weight']} "
        f"margin={CONFIG['rand_neg_margin']} t_max={CONFIG['rand_neg_t_max']}"
    )
if CONFIG["decoder_aux_weight"] > 0:
    print(f"decoder-aux: weight={CONFIG['decoder_aux_weight']} t_max={CONFIG['decoder_aux_t_max']}")

/content/ired-reasoning/ired/energy_net.py:90: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)


TypeError: GaussianLatentDiffusion.__init__() got an unexpected keyword argument 'rand_neg_t_max'

In [ ]:
opt = AdamW(ebm.parameters(), lr=CONFIG["lr_ebm"], weight_decay=CONFIG["weight_decay"])
step = 0
train_iter = iter(ebm_train_loader)
t0 = time.time()
accum = max(CONFIG["grad_accum_steps"], 1)
print(f"effective batch size: {CONFIG['batch_size']} x {accum} = {CONFIG['batch_size'] * accum}")

while step < CONFIG["steps_ebm"]:
    opt.zero_grad()
    loss_sum = 0.0
    stats_accum: dict[str, float] = {}
    for _ in range(accum):
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(ebm_train_loader)
            batch = next(train_iter)

        with torch.no_grad():
            z_q = ae.encode_to_latents(batch["question"], CONFIG["device"], max_length=CONFIG["max_q_length"])
            z_a = ae.encode_to_latents(batch["answer"],   CONFIG["device"], max_length=CONFIG["max_a_length"])

        loss, stats = diffusion(
            z_q, z_a,
            gold_texts=batch["answer"] if CONFIG["decoder_aux_weight"] > 0 else None,
        )
        if not torch.isfinite(loss):
            raise RuntimeError(f"step {step}: non-finite loss {loss.item()} (stats={stats})")

        (loss / accum).backward()
        loss_sum += loss.item()
        for k, v in stats.items():
            stats_accum[k] = stats_accum.get(k, 0.0) + float(v)

    torch.nn.utils.clip_grad_norm_(ebm.parameters(), 1.0)
    opt.step()

    step += 1
    avg_loss = loss_sum / accum
    avg_stats = {k: v / accum for k, v in stats_accum.items()}
    if step % CONFIG["log_every"] == 0:
        extras = " | ".join(f"{k} {v:.4f}" for k, v in avg_stats.items())
        print(f"[ebm] step {step:6d} | loss {avg_loss:.4f} | {extras} | {time.time()-t0:.1f}s")

    if step % CONFIG["eval_every"] == 0 or step == CONFIG["steps_ebm"]:
        ev_mbpp = eval_corpus(
            ae, diffusion, mbpp_test_ds, "mbpp", CONFIG["device"],
            CONFIG["max_q_length"], CONFIG["max_a_length"], CONFIG["inner_steps"],
            n_examples=CONFIG["eval_n_examples"], batch_size=CONFIG["batch_size"],
        )
        ev_he = eval_corpus(
            ae, diffusion, he_test_ds, "humaneval", CONFIG["device"],
            CONFIG["max_q_length"], CONFIG["max_a_length"], CONFIG["inner_steps"],
            n_examples=CONFIG["eval_n_examples"], batch_size=CONFIG["batch_size"],
        )
        print(
            f"  [mbpp n={ev_mbpp['n']}] pass(inner={CONFIG['inner_steps']})={ev_mbpp['acc']:.3f}  "
            f"pass(inner=0)={ev_mbpp['acc_inner0']:.3f}  ae={ev_mbpp['ae_acc']:.3f}"
        )
        print(
            f"  [humaneval n={ev_he['n']}] pass(inner={CONFIG['inner_steps']})={ev_he['acc']:.3f}  "
            f"pass(inner=0)={ev_he['acc_inner0']:.3f}  ae={ev_he['ae_acc']:.3f}"
        )
        print(
            f"  [latent mbpp] mse_z={ev_mbpp['mse_z']:.3f}  mse_z(inner=0)={ev_mbpp['mse_z_inner0']:.3f}  "
            f"std_za={ev_mbpp['std_za']:.3f}  std_zs={ev_mbpp['std_zs']:.3f}  corr_z={ev_mbpp['corr_z']:+.3f}"
        )
        # Exposure-bias gap (docs/exposure_bias.md): if both CEs are low but
        # pass-rate is also low, the AR decoder is leaking cognition.
        print(
            f"  [exposure mbpp] ce_pass={ev_mbpp['ce_pass']:.3f} (n={ev_mbpp['n_pass']})  "
            f"ce_fail={ev_mbpp['ce_fail']:.3f} (n={ev_mbpp['n_fail']})  "
            f"gap={ev_mbpp['eb_gap']:+.3f}"
        )
        print(
            f"  [exposure humaneval] ce_pass={ev_he['ce_pass']:.3f} (n={ev_he['n_pass']})  "
            f"ce_fail={ev_he['ce_fail']:.3f} (n={ev_he['n_fail']})  "
            f"gap={ev_he['eb_gap']:+.3f}"
        )
        ckpt = {
            "ebm": ebm.state_dict(), "config": CONFIG, "step": step,
            "mbpp_pass": ev_mbpp["acc"], "humaneval_pass": ev_he["acc"],
            "mbpp_ae_pass": ev_mbpp["ae_acc"], "humaneval_ae_pass": ev_he["ae_acc"],
            "mse_z_mbpp": ev_mbpp["mse_z"],
            "mbpp_eb_gap": ev_mbpp["eb_gap"], "humaneval_eb_gap": ev_he["eb_gap"],
        }
        torch.save(ckpt, os.path.join(CONFIG["ebm_dir"], f"ebm_step{step}.pt"))
        torch.save(ckpt, os.path.join(CONFIG["ebm_dir"], "ebm_latest.pt"))

print(f"\nMilestone 2 done. mbpp_pass={ev_mbpp['acc']:.3f}  humaneval_pass={ev_he['acc']:.3f}")


effective batch size: 4 x 8 = 32
[ebm] step     50 | loss 11.3731 | mse 0.9744 | nce 0.0000 | opt 4.6411 | e_real 107378.2275 | e_fake 120023.5693 | eps_scale 0.2573 | rand 9.3958 | e_rand 124830.5146 | dec 1.0029 | dec_n 0.5000 | 647.7s
[ebm] step    100 | loss 1.7237 | mse 0.9979 | nce 0.0000 | opt 5.3606 | e_real 117251.6982 | e_fake 125707.9199 | eps_scale 0.2642 | rand 0.0000 | e_rand 131821.2637 | dec 0.7258 | dec_n 0.5000 | 1287.3s
[ebm] step    150 | loss 1.3099 | mse 1.0087 | nce 0.0000 | opt 4.9871 | e_real 112882.5977 | e_fake 119340.1016 | eps_scale 0.2444 | rand 0.0000 | e_rand 125219.7871 | dec 0.3012 | dec_n 0.3750 | 1930.6s
[ebm] step    200 | loss 9.9759 | mse 0.9914 | nce 7.4004 | opt 5.7471 | e_real 81885.4990 | e_fake 88552.5273 | eps_scale 0.2117 | rand 0.6199 | e_rand 93411.4639 | dec 0.9643 | dec_n 0.5000 | 2573.8s
[ebm] step    250 | loss 1.5546 | mse 1.0003 | nce 0.0000 | opt 4.4623 | e_real 136328.1016 | e_fake 144016.4902 | eps_scale 0.2463 | rand 0.0000 | e_

: 

## Milestone 3 — Test-time compute sweep

Sweep `inner_steps ∈ {0, 1, 2, 5, 10}` on the test set and report pass-rate (execute decoded code against per-example tests).

**Decision rule (gensis §8):** a steeper pass-rate-vs-compute curve than AR-CoT at matched FLOPs validates the thesis. A flat or decreasing curve is an informative negative about smoothness of reasoning in latent space.

In [ ]:
diffusion.eval()
sweep = [0, 1, 2, 5, 10]
n_eval_examples = min(160, len(mbpp_test_ds))
results = []

print(f"sweeping inner_steps on MBPP test (n={n_eval_examples})")
print(f"{'inner':>6}  {'pass':>6}  {'n':>5}  {'ebm_passes':>11}  {'time_s':>7}")
for n_inner in sweep:
    t = time.time()
    correct = total = 0
    with torch.no_grad():
        for start in range(0, n_eval_examples, CONFIG["batch_size"]):
            batch_idx = list(range(start, min(start + CONFIG["batch_size"], n_eval_examples)))
            batch = [mbpp_test_ds[i] for i in batch_idx]
            questions = [b["question"] for b in batch]
            z_q = ae.encode_to_latents(
                questions, CONFIG["device"], max_length=CONFIG["max_q_length"],
            )
            z = diffusion.sample(z_q, inner_steps=n_inner)
            preds = ae.decode(z, max_length=CONFIG["max_a_length"])
            for ex, pred in zip(batch, preds):
                total += 1
                if verify_code(pred, ex):
                    correct += 1
    acc = correct / max(total, 1)
    dt = time.time() - t
    ebm_passes = CONFIG["timesteps"] * (1 + n_inner)
    print(f"{n_inner:>6}  {acc:>.3f}  {total:>5}  {ebm_passes:>11}  {dt:>7.1f}")
    results.append({"inner": n_inner, "acc": acc, "n": total, "ebm_passes": ebm_passes, "time_s": dt})


### Optional: plot the curve

Requires `matplotlib`. Skip if running headless.

In [ ]:
try:
    import matplotlib.pyplot as plt
    xs = [r["ebm_passes"] for r in results]
    ys = [r["acc"] for r in results]
    plt.figure(figsize=(6, 4))
    plt.plot(xs, ys, marker="o")
    plt.xlabel("EBM forward+backward passes per sample")
    plt.ylabel("final-answer accuracy")
    plt.title("Test-time compute curve (IRED in latent space)")
    plt.grid(True, alpha=0.3)
    plt.show()
except ImportError:
    print("matplotlib not installed; skip plot.")

## What the result tells you

- **Milestone 1 < ~90% extraction** → the pool is the bottleneck; the EBM can't recover what's lost here. Train longer / bigger pool / fewer compression (smaller `K` doesn't help — *larger* would).
- **Milestone 2 `eps_scale` drifts away from 1.0** → head-magnitude inflation. Increase `weight_decay`, lower `lr`, or check for any change to the energy parameterization.
- **Milestone 2 `corr_z ≈ 0`** → EBM hasn't learned a useful denoising direction yet. Train longer or revisit the contrast strength.
- **Milestone 2 `std_zs >> std_za`** → reverse process inflates magnitude. Usually a downstream symptom of `eps_scale` drift.
- **Milestone 3 curve flat / decreasing in `inner_steps`** → either the energy landscape isn't calibrated (NCE collapsed) or `opt_step` is over-stepping. Try lower `opt_step_size` or check `e_real/e_fake` history.
- **Curve monotonically rising with `inner_steps`** → IRED's thesis is validating. Compare against AR-CoT at matched FLOPs.